### NB2 — Molecular Descriptor Generation
### Author: Hamid Bouseber
### Project: COX-2 QSAR-XAI workflow
### Input: curated train/test and external datasets from NB1
### Output: descriptor matrices for PubChem, Morgan, and RDKit

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

from padelpy import from_smiles

In [2]:
# ----------------------------
# Configuration
# ----------------------------
DATA_DIR = "./data"
MATRICES_DIR = "./matrices"
os.makedirs(MATRICES_DIR, exist_ok=True)

TRAIN_TEST_FILE = f"{DATA_DIR}/dataset_train_test_molecule.csv"
EXTERNAL_FILE = f"{DATA_DIR}/dataset_external_molecule.csv"

SMILES_COL = "canonical_smiles"
TARGET_COL = "pIC50"

MORGAN_RADIUS = 2
MORGAN_N_BITS = 1024

PUBCHEM_BATCH_SIZE = 200

In [3]:
df_train_test = pd.read_csv(TRAIN_TEST_FILE)
df_external = pd.read_csv(EXTERNAL_FILE)

print("Train/test:", df_train_test.shape)
print("External:", df_external.shape)

assert SMILES_COL in df_train_test.columns, f"Missing column: {SMILES_COL}"
assert SMILES_COL in df_external.columns, f"Missing column: {SMILES_COL}"
assert TARGET_COL in df_train_test.columns, f"Missing column: {TARGET_COL}"
assert TARGET_COL in df_external.columns, f"Missing column: {TARGET_COL}"

assert len(df_train_test) > 0, "Train/test dataset is empty."
assert len(df_external) > 0, "External dataset is empty."

Train/test: (2297, 9)
External: (58, 5)


In [4]:
y_train_test = df_train_test[[TARGET_COL]].copy()
y_external = df_external[[TARGET_COL]].copy()

y_train_test.to_csv(f"{MATRICES_DIR}/y_train_test.csv", index=False)
y_external.to_csv(f"{MATRICES_DIR}/y_external.csv", index=False)

print("Saved target vectors:")
print("y_train_test:", y_train_test.shape)
print("y_external:", y_external.shape)

Saved target vectors:
y_train_test: (2297, 1)
y_external: (58, 1)


In [5]:
def mol_from_smiles(smiles):
    return Chem.MolFromSmiles(str(smiles))


def clean_numeric_matrix(X_train, X_external):
    """
    Replace inf values, impute missing values using train medians,
    and apply the same treatment to the external set.
    """
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_external = X_external.replace([np.inf, -np.inf], np.nan)

    medians = X_train.median(numeric_only=True)

    X_train = X_train.fillna(medians)
    X_external = X_external.fillna(medians)

    X_train = X_train.fillna(0)
    X_external = X_external.fillna(0)

    return X_train, X_external


def remove_useless_columns(X_train, X_external):
    """
    Remove non-informative columns based only on the train/test matrix:
    - constant columns
    - duplicated columns

    The same retained columns are then applied to the external matrix.
    """
    # Remove constant columns
    nunique = X_train.nunique(dropna=False)
    keep_cols = nunique[nunique > 1].index.tolist()

    X_train = X_train[keep_cols].copy()
    X_external = X_external[keep_cols].copy()

    # Remove duplicated columns
    duplicated_cols = X_train.T.duplicated()
    keep_cols = X_train.columns[~duplicated_cols].tolist()

    X_train = X_train[keep_cols].copy()
    X_external = X_external[keep_cols].copy()

    return X_train, X_external

# PubChem fingerprint generation

In [6]:
def compute_pubchem_fp(smiles_list, batch_size=200):
    all_rows = []

    for start in tqdm(range(0, len(smiles_list), batch_size), desc="PubChemFP"):
        batch = smiles_list[start:start + batch_size]

        try:
            fps = from_smiles(
                batch,
                fingerprints=True,
                descriptors=False
            )
        except Exception:
            fps = [{} for _ in batch]

        if isinstance(fps, dict):
            fps = [fps]

        for fp in fps:
            row = []
            for i in range(881):
                value = fp.get(f"PubchemFP{i}", 0)
                try:
                    row.append(int(value))
                except Exception:
                    row.append(0)
            all_rows.append(row)

        while len(all_rows) < start + len(batch):
            all_rows.append([0] * 881)

    cols = [f"PubchemFP{i}" for i in range(881)]
    return pd.DataFrame(all_rows, columns=cols)

In [7]:
X_pubchem_train_test = compute_pubchem_fp(
    df_train_test[SMILES_COL].tolist(),
    batch_size=PUBCHEM_BATCH_SIZE
)

X_pubchem_external = compute_pubchem_fp(
    df_external[SMILES_COL].tolist(),
    batch_size=PUBCHEM_BATCH_SIZE
)

X_pubchem_train_test, X_pubchem_external = remove_useless_columns(
    X_pubchem_train_test,
    X_pubchem_external
)

X_pubchem_train_test.to_csv(f"{MATRICES_DIR}/X_pubchem_train_test.csv", index=False)
X_pubchem_external.to_csv(f"{MATRICES_DIR}/X_pubchem_external.csv", index=False)

print("PubChemFP:", X_pubchem_train_test.shape, X_pubchem_external.shape)

PubChemFP: 100%|███████████████████████████████████████████████████████| 1/1 [00:22<00:00, 22.06s/it]


PubChemFP: (2297, 481) (58, 481)


# Morgan fingerprint generation

In [8]:
def compute_morgan_fp(smiles_list, radius=2, n_bits=2048):
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=radius,
        fpSize=n_bits
    )

    rows = []

    for smi in tqdm(smiles_list, desc="MorganFP"):
        mol = mol_from_smiles(smi)

        if mol is None:
            rows.append([0] * n_bits)
            continue

        fp = generator.GetFingerprint(mol)
        arr = np.zeros((n_bits,), dtype=int)
        arr[list(fp.GetOnBits())] = 1

        rows.append(arr.tolist())

    cols = [f"MorganFP{i}" for i in range(n_bits)]
    return pd.DataFrame(rows, columns=cols)

In [9]:
X_morgan_train_test = compute_morgan_fp(
    df_train_test[SMILES_COL].tolist(),
    radius=MORGAN_RADIUS,
    n_bits=MORGAN_N_BITS
)

X_morgan_external = compute_morgan_fp(
    df_external[SMILES_COL].tolist(),
    radius=MORGAN_RADIUS,
    n_bits=MORGAN_N_BITS
)

X_morgan_train_test, X_morgan_external = remove_useless_columns(
    X_morgan_train_test,
    X_morgan_external
)

X_morgan_train_test.to_csv(f"{MATRICES_DIR}/X_morgan_train_test.csv", index=False)
X_morgan_external.to_csv(f"{MATRICES_DIR}/X_morgan_external.csv", index=False)

print("MorganFP:", X_morgan_train_test.shape, X_morgan_external.shape)

MorganFP: 100%|████████████████████████████████████████████████████| 58/58 [00:00<00:00, 2161.17it/s]


MorganFP: (2297, 1020) (58, 1020)


# RDKit descriptor calculation

In [10]:
rdkit_descriptor_list = Descriptors._descList
rdkit_descriptor_names = [name for name, func in rdkit_descriptor_list]

print("Initial RDKit descriptors:", len(rdkit_descriptor_names))

Initial RDKit descriptors: 217


In [11]:
def compute_rdkit_descriptors(smiles_list):
    rows = []

    for smi in tqdm(smiles_list, desc="RDKit descriptors"):
        mol = mol_from_smiles(smi)

        if mol is None:
            rows.append([np.nan] * len(rdkit_descriptor_list))
            continue

        values = []

        for name, func in rdkit_descriptor_list:
            try:
                values.append(func(mol))
            except Exception:
                values.append(np.nan)

        rows.append(values)

    return pd.DataFrame(rows, columns=rdkit_descriptor_names)

In [12]:
X_rdkit_train_test = compute_rdkit_descriptors(
    df_train_test[SMILES_COL].tolist()
)

X_rdkit_external = compute_rdkit_descriptors(
    df_external[SMILES_COL].tolist()
)

X_rdkit_train_test, X_rdkit_external = clean_numeric_matrix(
    X_rdkit_train_test,
    X_rdkit_external
)

X_rdkit_train_test, X_rdkit_external = remove_useless_columns(
    X_rdkit_train_test,
    X_rdkit_external
)

X_rdkit_train_test.to_csv(f"{MATRICES_DIR}/X_rdkit_train_test.csv", index=False)
X_rdkit_external.to_csv(f"{MATRICES_DIR}/X_rdkit_external.csv", index=False)

print("RDKit:", X_rdkit_train_test.shape, X_rdkit_external.shape)

RDKit descriptors: 100%|█████████████████████████████████████████████| 58/58 [00:00<00:00, 76.73it/s]


RDKit: (2297, 195) (58, 195)


# Descriptor cleaning and export

In [13]:
expected_files = [
    "X_pubchem_train_test.csv",
    "X_pubchem_external.csv",
    "X_morgan_train_test.csv",
    "X_morgan_external.csv",
    "X_rdkit_train_test.csv",
    "X_rdkit_external.csv",
    "y_train_test.csv",
    "y_external.csv",
]

for file in expected_files:
    path = f"{MATRICES_DIR}/{file}"
    assert os.path.exists(path), f"Missing file: {path}"

for desc in ["pubchem", "morgan", "rdkit"]:
    X_train = pd.read_csv(f"{MATRICES_DIR}/X_{desc}_train_test.csv")
    X_ext = pd.read_csv(f"{MATRICES_DIR}/X_{desc}_external.csv")

    assert X_train.shape[1] == X_ext.shape[1], f"Feature number mismatch: {desc}"
    assert list(X_train.columns) == list(X_ext.columns), f"Column mismatch: {desc}"

    assert len(X_train) == len(y_train_test), f"Row mismatch train/test: {desc}"
    assert len(X_ext) == len(y_external), f"Row mismatch external: {desc}"

print("NB2 completed successfully.")
print("All descriptor matrices are aligned and saved in:", MATRICES_DIR)

NB2 completed successfully.
All descriptor matrices are aligned and saved in: ./matrices


In [14]:
summary = []

for desc in ["pubchem", "morgan", "rdkit"]:
    X_train = pd.read_csv(f"{MATRICES_DIR}/X_{desc}_train_test.csv")
    X_ext = pd.read_csv(f"{MATRICES_DIR}/X_{desc}_external.csv")

    summary.append({
        "descriptor": desc,
        "train_test_rows": X_train.shape[0],
        "external_rows": X_ext.shape[0],
        "features": X_train.shape[1],
    })

pd.DataFrame(summary)

,descriptor,train_test_rows,external_rows,features
0,pubchem,2297,58,481
1,morgan,2297,58,1020
2,rdkit,2297,58,195
